# Stage A3 — Normalization & Train/Validation/Test Split

Dual-fuel PINN pipeline | Sandrine Schueller Mafra | PPGEM – UFPR
Supports dissertation Section 3.1.3.2 (feedback item asking to justify
the 70-15-15 split), Section 3.1.2.3 (Eq. 3.1, normalization).

Self-contained: reads `masters_data.xlsx` directly, no dependency on
A1/A2 outputs, no thesis-text reference values (this stage isn't a
text-vs-data check, so there's nothing here to compare against).

**Input:** `data/masters_data.xlsx`
**Output:** every figure and result table is saved to `outputs/html/` as
`A3_<section>[_qualifier].html`. The `split` label (train/val/test), the
min-max normalized columns and the train-only scaling parameters are
saved as CSV only if you run the optional save cell at the end (those
CSVs are read by code, so they stay CSV).

**Split design in one sentence:** every row gets an *extremity score*
(how close it sits to the min or max of whichever input was swept for
it, Sec. 3.1.1.2's OFAT design), and the 6 highest-extremity rows
become the test set, the next 6 become validation, and the rest — the
interior/bulk of the design — become train. That directly targets the
"stratified split preserving extremes in test" requirement instead of
a plain random 70/15/15.

## Setup

In [1]:
import numpy as np
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

print("polars ", pl.__version__)
import plotly
print("plotly ", plotly.__version__)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW_PATH = PROJECT_ROOT / "data" / "masters_data.xlsx"
OUT_DIR = PROJECT_ROOT / "outputs"

SEED = 42  # fixes tie-breaking below so the split is reproducible run to run
RAW_PATH


polars  1.43.2
plotly  6.9.0


WindowsPath('c:/Users/sandr/Documents/GitHub/Master-s-Thesis/data/masters_data.xlsx')

## Color palette and output naming (shared across the whole pipeline)

`SPLIT_COLORS` (semantic role: train / validation / test / reference
value / alert / neutral) and `VARIABLE_COLORS` (identity of each of the
4 inputs and 5 outputs) are identical in every A/B/C notebook, so the
same element always has the same color in any chart of the pipeline.

Every figure or result table generated below is also saved to
`outputs/html/`, named `A3_<section>[_qualifier].html` — the number
matches the corresponding section header, so the order in which each
output was produced can be read from the file name alone.

In [2]:
SPLIT_COLORS = {
    "train": "#B7C9DA",
    "validation": "#2B6EFF",
    "test": "#571D99",
    "reference": "#343A40",   # value transcribed from the dissertation text
    "alert": "#E85D04",       # outlier / out of range / anomaly
    "neutral": "#B0AFA8",     # grid lines / neutral reference
}
VARIABLE_COLORS = {
    "SOI": "#073b3a", "lambda": "#0b6e4f", "sub_rate": "#08a045", "P_rail": "#6bbf59",
    "NOx": "#c7adff", "PM": "#916dd5", "eta": "#7151a9", "HC": "#573d7f", "CO2": "#46325d",
}

HTML_DIR = OUT_DIR / "html"
HTML_DIR.mkdir(parents=True, exist_ok=True)


def flagged_table_html(df, title, out_path, flag_col=None, is_flagged=lambda v: False, ref_cols=()):
    """Result table -> Plotly go.Table -> HTML.
    Columns listed in ref_cols get the 'reference' tone in the header
    (values transcribed from the dissertation text). Cells in flag_col
    get the 'alert' tone wherever is_flagged(value) is True."""
    cols = list(df.columns)
    n = df.shape[0]
    header_fill = [SPLIT_COLORS["reference"] if c in ref_cols else "#F1F3F5" for c in cols]
    header_font = ["white" if c in ref_cols else "black" for c in cols]
    cell_fill = []
    for c in cols:
        if c == flag_col:
            cell_fill.append([SPLIT_COLORS["alert"] if is_flagged(v) else "white"
                               for v in df[c].to_list()])
        else:
            cell_fill.append(["white"] * n)
    fig = go.Figure(data=[go.Table(
        header=dict(values=cols, fill_color=header_fill,
                     font=dict(color=header_font), align="left"),
        cells=dict(values=[df[c].to_list() for c in cols],
                    fill_color=cell_fill, align="left"),
    )])
    fig.update_layout(title=title, margin=dict(t=40, l=10, r=10, b=10))
    fig.write_html(str(out_path), include_plotlyjs="inline")
    return fig


def simple_table_html(df, title, out_path):
    return flagged_table_html(df, title, out_path)

# This notebook labels the validation split "val" (the label every later
# stage filters on), so map the split labels to the shared palette here
# instead of renaming the label itself.
SPLIT_COLOR_OF = {"train": SPLIT_COLORS["train"], "val": SPLIT_COLORS["validation"],
                  "test": SPLIT_COLORS["test"]}
SPLIT_NAME_OF = {"train": "train", "val": "validation", "test": "test"}

## 1. Load data

In [3]:
COLUMN_MAP = {
    "SOI [o.CA]": "SOI", "Lambda [-]": "lambda", "Sub. Rate [%]": "sub_rate",
    "Prail [bar]": "P_rail", "HC [g/kW.h]": "HC", "NOX [ppm]": "NOx",
    "CO2 [%]": "CO2", "SO_H [FSN]": "PM", "ETA [%]": "eta",
}
INPUT_COLS = ["SOI", "lambda", "sub_rate", "P_rail"]
OUTPUT_COLS = ["HC", "NOx", "CO2", "PM", "eta"]
ALL_COLS = INPUT_COLS + OUTPUT_COLS

df = pl.read_excel(RAW_PATH).rename(COLUMN_MAP).select(ALL_COLS)
n = df.shape[0]
print(df.shape)
df.head()


(40, 9)


SOI,lambda,sub_rate,P_rail,HC,NOx,CO2,PM,eta
f64,f64,f64,f64,f64,f64,f64,f64,f64
10.041504,1.475656,67.606903,1599.775146,6.766725,1546.245605,7.921363,0.026,0.389684
9.030762,1.48207,67.867645,1600.039795,6.984949,1404.415405,7.911663,0.033,0.388466
8.02002,1.457225,66.625893,1599.994995,7.124145,1285.570679,7.906114,0.028,0.379428
7.009277,1.464357,66.948044,1600.072144,7.205558,1182.612427,7.900151,0.021,0.378484
5.998535,1.484915,68.046165,1599.916504,7.201936,1096.745239,7.913348,0.022,0.382072


## 2. Recompute OFAT block per row

Same data-driven method as A1 (deviation from each input's own median,
normalised by its range, with neighbour-smoothing) — repeated here
rather than imported, so this notebook has no dependency on A1 having
been run first.

In [4]:
medians = {c: df[c].median() for c in INPUT_COLS}
ranges = {c: (df[c].max() - df[c].min()) for c in INPUT_COLS}

deviation = np.column_stack([
    np.abs(df[c].to_numpy() - medians[c]) / ranges[c] for c in INPUT_COLS
])
raw_block = np.array(INPUT_COLS)[deviation.argmax(axis=1)]


def smooth_isolated_labels(labels, passes=2):
    out = list(labels)
    for _ in range(passes):
        changed = False
        for i in range(1, len(out) - 1):
            if out[i] != out[i - 1] and out[i - 1] == out[i + 1]:
                out[i] = out[i - 1]
                changed = True
        if not changed:
            break
    return np.array(out)


ofat_block = smooth_isolated_labels(raw_block)
df = df.with_columns(pl.Series("ofat_block", ofat_block))
print(np.unique(ofat_block, return_counts=True))


(array(['P_rail', 'SOI', 'lambda', 'sub_rate'], dtype='<U8'), array([ 3, 10, 16, 11]))


**Equation implemented above** — for input $c$ and row $i$:

$$
\text{deviation}_{i,c} = \frac{\left| x_{i,c} - \text{median}(x_{:,c}) \right|}{\max(x_{:,c}) - \min(x_{:,c})}
\qquad\Longrightarrow\qquad
\text{block}_i = \arg\max_{c} \; \text{deviation}_{i,c}
$$

i.e. whichever input sits furthest from its own dataset median, scaled
by its own range, is taken as the swept variable for that row.

In [25]:
block_names, block_sizes = np.unique(ofat_block, return_counts=True)
fig = go.Figure(go.Bar(
    x=list(block_names), y=list(block_sizes),
    marker_color=[VARIABLE_COLORS[b] for b in block_names],
))
fig.update_layout(title="Rows per inferred OFAT block", yaxis_title="count",
                   width=750, height=550)
fig.show()
fig.write_html(str(HTML_DIR / "A3_02_block_counts.html"), include_plotlyjs="inline")

## 3. Extremity score & split assignment

For each row, rank it within its own OFAT block by the value of that
block's swept variable, and convert the rank to a 0-1 position (0 = the
block's minimum, 1 = its maximum):

$$
\text{position}_i = \frac{\text{rank}_i}{m_b - 1}, \qquad
\text{extremity}_i = \left| \text{position}_i - 0.5 \right| \times 2
$$

where $m_b$ is the size of row $i$'s block — so a block's extremes
score 1.0 and its central point scores 0.0.

Sort **all 40 rows together** by extremity, descending. The top 6
become test, the next 6 become validation, the remaining 28 become
train — 70/15/15 by construction, and test is deliberately weighted
toward domain boundaries rather than a plain random draw.

Ties at extremity = 1.0 (several blocks' min/max compete for the same
few slots) are broken by a seeded random jitter, so the exact rows
selected are reproducible but not hand-picked.

In [6]:
extremity = np.zeros(n)
for b in np.unique(ofat_block):
    idx = np.where(ofat_block == b)[0]
    vals = df[b].to_numpy()[idx]
    order = np.argsort(vals)
    m = len(idx)
    if m == 1:
        pos = np.array([0.5])
    else:
        ranks = np.empty(m)
        ranks[order] = np.arange(m)
        pos = ranks / (m - 1)
    extremity[idx] = np.abs(pos - 0.5) * 2

rng = np.random.default_rng(SEED)
jitter = rng.uniform(-1e-9, 1e-9, size=n)
order = np.argsort(-(extremity + jitter))

split = np.array(["train"] * n)
split[order[:6]] = "test"
split[order[6:12]] = "val"

df = df.with_columns(pl.Series("extremity", extremity), pl.Series("split", split))

from collections import Counter
split_counts = Counter(split)
split_sizes = pl.DataFrame({
    "split": [SPLIT_NAME_OF[s] for s in ["train", "val", "test"]],
    "n": [split_counts[s] for s in ["train", "val", "test"]],
    "share": [round(split_counts[s] / n, 3) for s in ["train", "val", "test"]],
})
simple_table_html(split_sizes, "A3 -- Split sizes", HTML_DIR / "A3_03_split_sizes.html")
split_sizes

split,n,share
str,i64,f64
"""train""",28,0.7
"""validation""",6,0.15
"""test""",6,0.15


**Visual check** — the score that actually drove the split, one point per row:

In [24]:
row_ids = np.arange(n)

fig = go.Figure()
for s in ["train", "val", "test"]:
    mask = split == s
    fig.add_trace(go.Scatter(x=row_ids[mask], y=extremity[mask], mode="markers",
                              marker=dict(color=SPLIT_COLOR_OF[s], size=10,
                                          line=dict(color=SPLIT_COLORS["reference"], width=0.0)),
                              name=SPLIT_NAME_OF[s]))
fig.add_hline(y=extremity[order[11]], line_dash="dot", line_color=SPLIT_COLORS["neutral"],
              annotation_text="val/train cutoff")
fig.update_layout(title="Extremity score per row, coloured by resulting split",
                   xaxis_title="row index", yaxis_title="extremity score",
                   width=950, height=600)
fig.show()
fig.write_html(str(HTML_DIR / "A3_03_extremity_scatter.html"), include_plotlyjs="inline")

## 4. Check split composition — sizes and OFAT-block coverage

A block with **zero** rows in validation or test is flagged in the alert
color: the model is then never evaluated on that swept variable in that
split. This is a property of the extremity-based split applied to blocks
of very different sizes, not an error — but it limits what the
validation/test metrics can say about that input.

In [8]:
blocks = sorted(np.unique(ofat_block), key=lambda b: -int((ofat_block == b).sum()))
coverage_rows = []
for b in blocks:
    in_block = ofat_block == b
    counts = {s: int((in_block & (split == s)).sum()) for s in ["train", "val", "test"]}
    coverage_rows.append({
        "ofat_block": b, "train": counts["train"], "validation": counts["val"],
        "test": counts["test"], "total": int(in_block.sum()),
        "present_in_all_splits": all(v > 0 for v in counts.values()),
    })
block_coverage = pl.DataFrame(coverage_rows)
flagged_table_html(block_coverage, "A3 -- OFAT-block coverage per split", HTML_DIR / "A3_04_block_coverage.html",
                    flag_col="present_in_all_splits", is_flagged=lambda v: not v)
block_coverage

ofat_block,train,validation,test,total,present_in_all_splits
str,i64,i64,i64,i64,bool
"""lambda""",12,2,2,16,true
"""sub_rate""",7,2,2,11,true
"""SOI""",8,0,2,10,false
"""P_rail""",1,2,0,3,false


In [9]:
# Which split holds the row with the dataset's overall min / max of each
# input (not just its own block's)
extreme_rows = []
for c in INPUT_COLS:
    min_row = df.filter(pl.col(c) == pl.col(c).min())
    max_row = df.filter(pl.col(c) == pl.col(c).max())
    extreme_rows.append({
        "input": c,
        "global_min": round(min_row[c][0], 4), "split_of_global_min": SPLIT_NAME_OF[min_row["split"][0]],
        "global_max": round(max_row[c][0], 4), "split_of_global_max": SPLIT_NAME_OF[max_row["split"][0]],
    })
global_extremes = pl.DataFrame(extreme_rows)
simple_table_html(global_extremes, "A3 -- Split holding each input's global min / max",
                   HTML_DIR / "A3_04_global_extremes.html")
global_extremes

input,global_min,split_of_global_min,global_max,split_of_global_max
str,f64,str,f64,str
"""SOI""",0.9448,"""test""",10.0415,"""test"""
"""lambda""",1.1132,"""test""",1.6489,"""test"""
"""sub_rate""",0.7718,"""test""",91.7971,"""test"""
"""P_rail""",1299.6168,"""validation""",1800.0833,"""validation"""


## 5. Min-max normalization (Eq. 3.1) — fit on train only

$$
x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}
$$

$x_{\min}$/$x_{\max}$ are computed **from the training rows only** and
then applied to all three splits, so no information about the
validation or test rows leaks into the scaling — the requirement
Sec. 3.1.2.3 already states in words.

In [10]:
train = df.filter(pl.col("split") == "train")
train_min = {c: train[c].min() for c in ALL_COLS}
train_max = {c: train[c].max() for c in ALL_COLS}

norm_exprs = [
    ((pl.col(c) - train_min[c]) / (train_max[c] - train_min[c])).alias(f"{c}_norm")
    for c in ALL_COLS
]
df = df.with_columns(norm_exprs)

scaling = pl.DataFrame({
    "variable": ALL_COLS,
    "train_min": [round(train_min[c], 6) for c in ALL_COLS],
    "train_max": [round(train_max[c], 6) for c in ALL_COLS],
})
simple_table_html(scaling, "A3 -- Train-only scaling parameters (Eq. 3.1)",
                   HTML_DIR / "A3_05_scaling_params.html")
df.select(["split"] + [f"{c}_norm" for c in ALL_COLS]).head()

split,SOI_norm,lambda_norm,sub_rate_norm,P_rail_norm,HC_norm,NOx_norm,CO2_norm,PM_norm,eta_norm
str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""test""",1.142857,0.794905,0.721092,0.666102,0.47648,1.202255,0.249685,0.077465,1.024328
"""train""",1.0,0.815269,0.725503,0.666984,0.507262,1.0,0.245944,0.126761,1.0
"""train""",0.857143,0.736396,0.704495,0.666834,0.526897,0.830523,0.243804,0.091549,0.819438
"""train""",0.714286,0.759037,0.709945,0.667092,0.538381,0.683701,0.241504,0.042254,0.800565
"""train""",0.571429,0.8243,0.728523,0.666573,0.53787,0.561252,0.246594,0.049296,0.872258


**Before vs. after**, raw units on the left and the same points normalized on the right, for every variable:

In [23]:
fig = make_subplots(rows=3, cols=3, subplot_titles=[f"{c} (raw -> norm)" for c in ALL_COLS])

for i, c in enumerate(ALL_COLS):
    r, cc = divmod(i, 3)
    for s in ["train", "val", "test"]:
        sub = df.filter(pl.col("split") == s)
        fig.add_trace(
            go.Scatter(x=sub[c].to_numpy(), y=sub[f"{c}_norm"].to_numpy(), mode="markers",
                       marker=dict(color=SPLIT_COLOR_OF[s], size=7,
                                   line=dict(color=SPLIT_COLORS["reference"], width=0.0)),
                       name=SPLIT_NAME_OF[s], showlegend=(i == 0)),
            row=r + 1, col=cc + 1,
        )
    fig.add_hline(y=0, line_dash="dot", line_color=SPLIT_COLORS["neutral"], row=r + 1, col=cc + 1)
    fig.add_hline(y=1, line_dash="dot", line_color=SPLIT_COLORS["neutral"], row=r + 1, col=cc + 1)

fig.update_layout(height=1050, width=1150,
                   title_text="Raw value (x-axis) vs. train-normalized value (y-axis)")
fig.show()
fig.write_html(str(HTML_DIR / "A3_05_raw_vs_norm.html"), include_plotlyjs="inline")

## 6. Where val/test normalized values land outside [0, 1]

Because test/val intentionally hold the domain's boundary points,
normalizing by the (narrower) train-only range will legitimately push
some of those values below 0 or above 1. That's expected here, not a
bug — it's the model being asked to handle points slightly beyond what
it was fit on, which is exactly what a boundary-aware test set is for.
Worth flagging explicitly rather than discovering it silently later.

In [12]:
rows = []
for c in ALL_COLS:
    col = f"{c}_norm"
    non_train = df.filter(pl.col("split") != "train")
    below = non_train.filter(pl.col(col) < 0).shape[0]
    above = non_train.filter(pl.col(col) > 1).shape[0]
    rows.append({"variable": c, "below_0": below, "above_1": above, "n_outside": below + above,
                 "val_test_norm_min": round(non_train[col].min(), 3),
                 "val_test_norm_max": round(non_train[col].max(), 3),
                 "train_min": round(train_min[c], 4), "train_max": round(train_max[c], 4)})
range_check = pl.DataFrame(rows)
flagged_table_html(range_check, "A3 -- Val/test points outside [0, 1] after train-only normalization",
                    HTML_DIR / "A3_06_out_of_range_table.html",
                    flag_col="n_outside", is_flagged=lambda v: v > 0)
range_check

variable,below_0,above_1,n_outside,val_test_norm_min,val_test_norm_max,train_min,train_max
str,i64,i64,i64,f64,f64,f64,f64
"""SOI""",1,1,2,-0.143,1.143,1.9556,9.0308
"""lambda""",2,2,4,-0.356,1.345,1.2253,1.5403
"""sub_rate""",2,2,4,-0.41,1.13,24.9848,84.0926
"""P_rail""",1,1,2,-0.334,1.334,1399.907,1699.9635
"""HC""",4,2,6,-0.468,1.625,3.3888,10.4781
"""NOx""",3,1,4,-0.047,1.202,703.1699,1404.4154
"""CO2""",2,3,5,-0.247,1.323,7.2739,9.8669
"""PM""",2,2,4,-0.028,2.782,0.015,0.157
"""eta""",0,1,1,0.193,1.024,0.3384,0.3885


**Same table, as a picture** — how many val/test points fall below 0 or above 1 after train-only scaling:

In [22]:
fig = go.Figure()
fig.add_trace(go.Bar(x=range_check["variable"], y=range_check["below_0"],
                      name="below 0", marker_color=SPLIT_COLORS["alert"]))
fig.add_trace(go.Bar(x=range_check["variable"], y=range_check["above_1"],
                      name="above 1", marker_color=SPLIT_COLORS["alert"],
                      marker_pattern_shape="/", marker_pattern_fgcolor="white"))
fig.update_layout(barmode="group", title="Val/test points outside [0,1] after train-only normalization",
                   yaxis_title="count", width=950, height=600)
fig.show()
fig.write_html(str(HTML_DIR / "A3_06_out_of_range_bar.html"), include_plotlyjs="inline")

## 7. Train vs. validation vs. test per variable

Smooth KDE curve for train (n=28 supports a reasonable density
estimate); validation and test are shown as rug ticks at their actual
values rather than their own KDE curves, since a density estimate from
only 6 points would be more noise than signal.

In [21]:
from scipy.stats import gaussian_kde

fig = make_subplots(rows=3, cols=3, subplot_titles=ALL_COLS)

for i, col in enumerate(ALL_COLS):
    r, c = divmod(i, 3)
    train_vals = df.filter(pl.col("split") == "train")[col].to_numpy()
    val_vals = df.filter(pl.col("split") == "val")[col].to_numpy()
    test_vals = df.filter(pl.col("split") == "test")[col].to_numpy()

    kde = gaussian_kde(train_vals)
    x_grid = np.linspace(df[col].min(), df[col].max(), 200)
    density = kde(x_grid)
    peak = density.max()

    fig.add_trace(go.Scatter(x=x_grid, y=density, mode="lines",
                              line=dict(color=SPLIT_COLOR_OF["train"], width=2.5),
                              name="train", showlegend=(i == 0)),
                  row=r + 1, col=c + 1)
    for s, vals, offset in [("val", val_vals, -0.06), ("test", test_vals, -0.14)]:
        fig.add_trace(go.Scatter(x=vals, y=[offset * peak] * len(vals), mode="markers",
                                  marker=dict(color=SPLIT_COLOR_OF[s], symbol="line-ns", size=11,
                                              line=dict(width=2, color=SPLIT_COLOR_OF[s])),
                                  name=SPLIT_NAME_OF[s], showlegend=(i == 0)),
                      row=r + 1, col=c + 1)

fig.update_layout(height=1050, width=1000,
                   title_text="Train KDE with validation/test rug ticks, per variable")
fig.show()
fig.write_html(str(HTML_DIR / "A3_07_split_kde.html"), include_plotlyjs="inline")

## 8. Split assignment over the OFAT structure (extends A1's plot)

In [ ]:
row_ids = np.arange(n)
split_arr = df["split"].to_numpy()

fig = make_subplots(rows=2, cols=2, subplot_titles=INPUT_COLS)
for i, col in enumerate(INPUT_COLS):
    r, c = divmod(i, 2)
    col_vals = df[col].to_numpy()
    for s in ["train", "val", "test"]:
        mask = split_arr == s
        fig.add_trace(
            go.Scatter(x=row_ids[mask], y=col_vals[mask], mode="markers",
                       marker=dict(color=SPLIT_COLOR_OF[s], size=9,
                                   line=dict(color=SPLIT_COLORS["reference"], width=0.0)),
                       name=SPLIT_NAME_OF[s], showlegend=(i == 0)),
            row=r + 1, col=c + 1,
        )
fig.update_layout(height=850, width=1000,
                   title_text="Inputs vs. row index, coloured by split assignment")
fig.show()
fig.write_html(str(HTML_DIR / "A3_08_split_over_ofat.html"), include_plotlyjs="inline")

## 9. η output layer: sigmoid on the physical fraction + fixed rescaling (Sec. 3.2.1.4)

Sec. 3.2.1.4 and the "Bounds" row of Table 7 require the efficiency output
to satisfy 0 ≤ η ≤ 1 through a sigmoid activation. Every model from B1 to
C4 trains on the train-only min-max normalized targets of Section 5, so
the η output is built in two steps:

1. `Dense(1, activation="sigmoid")` produces η as a **physical fraction**,
   bounded to (0, 1) exactly as the text describes;
2. a fixed, non-trainable `Rescaling` layer applies the same Eq. 3.1
   scaling used for the targets, so the loss sees η on the same
   normalized scale as the other four outputs.

```python
eta_frac = layers.Dense(1, activation="sigmoid")(x)
eta_norm = layers.Rescaling(scale=1.0 / (eta_max - eta_min),
                            offset=-eta_min / (eta_max - eta_min))(eta_frac)
```

where `eta_min` / `eta_max` are the **train-only** η minimum and maximum
from Section 5, as a fraction. `Rescaling` has no trainable weights, so
the parameter counts of Table 6 are unchanged.

The cells below compute the constants B1–C4 must use, the interval the
rescaled output can take on the normalized scale, and two checks: that
every measured η lies inside the enforced bound, and that the sigmoid is
not saturated over the measured range (a saturated sigmoid would give
near-zero gradients and stall training of the η output).

In [16]:
eta_raw = df["eta"].to_numpy()
to_fraction = (lambda v: v / 100) if eta_raw.max() > 1 else (lambda v: v)   # A1 Section 7: stored as a fraction
eta_frac = to_fraction(eta_raw)
eta_min, eta_max = to_fraction(train_min["eta"]), to_fraction(train_max["eta"])
eta_span = eta_max - eta_min

rescale_scale = 1.0 / eta_span
rescale_offset = -eta_min / eta_span
norm_lower, norm_upper = 0 * rescale_scale + rescale_offset, 1 * rescale_scale + rescale_offset

n_outside = int(((eta_frac <= 0) | (eta_frac >= 1)).sum())
sig_slope = eta_frac * (1 - eta_frac)            # d sigmoid / dz = s (1 - s), max 0.25 at s = 0.5
saturated = bool(sig_slope.min() < 0.05)         # below ~20% of the maximum slope

eta_output_layer = pl.DataFrame([
    {"quantity": "eta_min (train, fraction)", "value": round(eta_min, 6),
     "note": "Rescaling constant", "status": "constant"},
    {"quantity": "eta_max (train, fraction)", "value": round(eta_max, 6),
     "note": "Rescaling constant", "status": "constant"},
    {"quantity": "Rescaling scale = 1 / (eta_max - eta_min)", "value": round(rescale_scale, 6),
     "note": "passed to layers.Rescaling(scale=...)", "status": "constant"},
    {"quantity": "Rescaling offset = -eta_min / (eta_max - eta_min)", "value": round(rescale_offset, 6),
     "note": "passed to layers.Rescaling(offset=...)", "status": "constant"},
    {"quantity": "normalized output lower bound (eta = 0)", "value": round(norm_lower, 4),
     "note": "interval the rescaled output can take", "status": "constant"},
    {"quantity": "normalized output upper bound (eta = 1)", "value": round(norm_upper, 4),
     "note": "interval the rescaled output can take", "status": "constant"},
    {"quantity": "measured eta points outside (0, 1), all splits", "value": float(n_outside),
     "note": f"of {len(eta_frac)} points", "status": "OK" if n_outside == 0 else "points outside bound"},
    {"quantity": "sigmoid slope over the measured eta range (min)", "value": round(float(sig_slope.min()), 4),
     "note": f"max {sig_slope.max():.4f}; theoretical maximum 0.25",
     "status": "saturated" if saturated else "OK"},
])
flagged_table_html(eta_output_layer, "A3 -- eta output layer: constants and checks",
                    HTML_DIR / "A3_09_eta_output_layer_table.html",
                    flag_col="status", is_flagged=lambda v: v not in ("constant", "OK"))
eta_output_layer

quantity,value,note,status
str,f64,str,str
"""eta_min (train, fraction)""",0.338413,"""Rescaling constant""","""constant"""
"""eta_max (train, fraction)""",0.388466,"""Rescaling constant""","""constant"""
"""Rescaling scale = 1 / (eta_max…",19.978917,"""passed to layers.Rescaling(sca…","""constant"""
"""Rescaling offset = -eta_min / …",-6.76113,"""passed to layers.Rescaling(off…","""constant"""
"""normalized output lower bound …",-6.7611,"""interval the rescaled output c…","""constant"""
"""normalized output upper bound …",13.2178,"""interval the rescaled output c…","""constant"""
"""measured eta points outside (0…",0.0,"""of 40 points""","""OK"""
"""sigmoid slope over the measure…",0.2239,"""max 0.2378; theoretical maximu…","""OK"""


In [19]:
# The sigmoid over its pre-activation z, with every measured eta placed at
# z = logit(eta) and coloured by split. Right axis: the same curve after the
# fixed rescaling, i.e. the normalized value the loss actually sees.
z = np.linspace(-6, 6, 400)
sig = 1 / (1 + np.exp(-z))
split_arr = df["split"].to_numpy()

fig = go.Figure()
fig.add_hrect(y0=eta_frac.min(), y1=eta_frac.max(), fillcolor=SPLIT_COLORS["neutral"],
              opacity=0.3, line_width=0, annotation_text="measured eta range",
              annotation_position="top left")
fig.add_trace(go.Scatter(x=z, y=sig, mode="lines", name="sigmoid(z) = eta",
                         line=dict(color=VARIABLE_COLORS["eta"], width=2.5)))
for s in ["train", "val", "test"]:
    mask = split_arr == s
    e = eta_frac[mask]
    fig.add_trace(go.Scatter(x=np.log(e / (1 - e)), y=e, mode="markers", name=SPLIT_NAME_OF[s],
                             marker=dict(color=SPLIT_COLOR_OF[s], size=9,
                                         line=dict(color=SPLIT_COLORS["reference"], width=0.0))))
fig.add_trace(go.Scatter(x=[None], y=[None], yaxis="y2", showlegend=False))  # makes the right axis render
fig.update_layout(
    title="eta output layer: sigmoid on the physical fraction (left) and after the fixed rescaling (right)",
    xaxis_title="pre-activation z",
    yaxis=dict(title="eta (fraction), bounded to (0, 1)", range=[0, 1]),
    yaxis2=dict(title="normalized eta seen by the loss", overlaying="y", side="right",
                range=[norm_lower, norm_upper], showgrid=False),
    width=900, height=450, legend=dict(x=0.02, y=0.98),
)
fig.show()
fig.write_html(str(HTML_DIR / "A3_09_eta_output_layer_sigmoid.html"), include_plotlyjs="inline")

## Optional — persist outputs

Saves row-level split/normalization info and the train-only scaling
parameters (needed again in every later stage, and in Phase D to
un-normalize predictions back to physical units).

In [18]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
df.write_csv(OUT_DIR / "A3_split_normalized.csv")

scaling = pl.DataFrame({
    "variable": ALL_COLS,
    "train_min": [train_min[c] for c in ALL_COLS],
    "train_max": [train_max[c] for c in ALL_COLS],
})
scaling.write_csv(OUT_DIR / "A3_scaling_params.csv")
print(f"Saved to {OUT_DIR}")


Saved to c:\Users\sandr\Documents\GitHub\Master-s-Thesis\outputs


## Next

**Phase B (B1 — architecture search)** and **Phase C (C1 — physics
constraints)** both branch from here: B1 needs `split` to build its
5-fold CV over the train+val rows, and C1's collocation sampling (LHS)
should stay inside the train-derived normalized range from Section 5
above, not the full dataset range.

Every model from B1 to C4 builds the η output as described in Section 9
(sigmoid on the physical fraction + fixed `Rescaling` with the train-only
η constants), so all later stages must compute `eta_min` / `eta_max`
from the same train rows used here.